<a href="https://colab.research.google.com/github/keyvanmahmoudi/Machine_Learning_Course/blob/main/solutions/Session10_model_selection_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۱۰: انتخاب مدل، تنظیم ابرپارامتر و دادهٔ نامتوازن

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session10_model_selection.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

داده و پیش‌پردازش جلسه را دوباره می‌سازیم:

</div>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import randint
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV,
                                     TunedThresholdClassifierCV)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import make_scorer, confusion_matrix
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_URL = "https://raw.githubusercontent.com/keyvanmahmoudi/Machine_Learning_Course/main/data/"
churn = pd.read_csv(DATA_URL + "Churn_Modelling.csv").drop(columns=["RowNumber", "CustomerId", "Surname"])
X, y = churn.drop(columns="Exited"), churn["Exited"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

numeric = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary"]
categorical = ["Geography", "Gender"]
preprocessor = ColumnTransformer([("num", StandardScaler(), numeric),
                                  ("cat", OneHotEncoder(drop="if_binary"), categorical)])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metrics = ["precision", "recall", "f1", "average_precision"]

def cv_summary(model):
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=metrics, n_jobs=-1)
    return {m: res[f"test_{m}"].mean() for m in metrics}

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

`RandomForestClassifier` را با `RandomizedSearchCV` روی همین داده تنظیم کنید و با HistGradientBoosting مقایسه کنید.

</div>

In [2]:
rf_search = RandomizedSearchCV(
    Pipeline([("prep", preprocessor), ("model", RandomForestClassifier(n_jobs=-1, random_state=42))]),
    {"model__n_estimators": randint(100, 400), "model__max_depth": [None, 6, 10, 15],
     "model__min_samples_leaf": randint(1, 20), "model__max_features": ["sqrt", 0.5, 1.0]},
    n_iter=15, cv=cv, scoring="average_precision", n_jobs=-1, random_state=42,
).fit(X_train, y_train)
print("Best random forest parameters:", rf_search.best_params_)

hgb = Pipeline([("prep", preprocessor), ("model", HistGradientBoostingClassifier(random_state=42))])
pd.DataFrame({"Random forest (tuned)": cv_summary(rf_search.best_estimator_),
              "HistGradientBoosting (default)": cv_summary(hgb)}).T.round(3)

Best random forest parameters: {'model__max_depth': 10, 'model__max_features': 0.5, 'model__min_samples_leaf': 16, 'model__n_estimators': 370}


,precision,recall,f1,average_precision
Random forest (tuned),0.772,0.450,0.569,0.689
HistGradientBoosting (default),0.733,0.498,0.593,0.692


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

جنگل تصادفی تنظیم‌شده به عملکردی نزدیک به HistGradientBoosting می‌رسد. HistGradientBoosting حتی با تنظیمات پیش‌فرض رقابتی است و بسیار سریع‌تر آموزش می‌بیند. در دادهٔ جدولی، این دو خانواده معمولاً بهترین‌اند و انتخاب بینشان بیشتر به سرعت، اندازهٔ داده و سادگی تنظیم بستگی دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

به‌جای SMOTE از `RandomUnderSampler` در `imblearn` استفاده کنید. نتیجه چه تفاوتی دارد؟

</div>

In [3]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

summary = {}
for name, sampler in [("none", None), ("SMOTE", SMOTE(random_state=42)),
                      ("RandomUnderSampler", RandomUnderSampler(random_state=42))]:
    steps = [("prep", preprocessor)] + ([("sampler", sampler)] if sampler else []) + \
            [("model", LogisticRegression(max_iter=1000))]
    summary[name] = cv_summary(ImbPipeline(steps))
pd.DataFrame(summary).T.round(3)

,precision,recall,f1,average_precision
none,0.597,0.214,0.315,0.469
SMOTE,0.383,0.681,0.490,0.456
RandomUnderSampler,0.380,0.683,0.488,0.454


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

Undersampling بخش بزرگی از مشتریان «نمانده» را دور می‌ریزد تا تعداد دو کلاس برابر شود. نتیجه شبیه SMOTE است: Recall بالا می‌رود و Precision پایین می‌آید، و Average Precision تقریباً ثابت می‌ماند. مزیت undersampling سرعت آموزش و نساختن دادهٔ مصنوعی است؛ عیبش دور ریختن اطلاعات، که با دادهٔ کم آسیب‌زا است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

فرض کنید تماس با هر مشتری ۱۰ واحد هزینه دارد و نگه داشتن یک مشتری ریزشی ۱۰۰ واحد سود. با یک تابع امتیاز سفارشی (`make_scorer`) آستانه‌ای را پیدا کنید که سود را بیشینه کند.

</div>

In [4]:
COST, GAIN = 10, 100

def retention_profit(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return GAIN * tp - COST * (tp + fp)

profit_scorer = make_scorer(retention_profit)
tuned = TunedThresholdClassifierCV(hgb, scoring=profit_scorer, cv=cv, n_jobs=-1).fit(X_train, y_train)
print(f"Profit-maximizing threshold (chosen by CV on training data): {tuned.best_threshold_:.3f}")

for name, model in [("threshold 0.5", hgb.fit(X_train, y_train)), ("profit-tuned threshold", tuned)]:
    pred = model.predict(X_test)
    print(f"{name:<24} calls = {pred.sum():4d}   test profit = {retention_profit(y_test, pred):,}")

Profit-maximizing threshold (chosen by CV on training data): 0.091


threshold 0.5            calls =  267   test profit = 16,730
profit-tuned threshold   calls =  985   test profit = 26,050


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با هزینهٔ ۱۰ و سود ۱۰۰، تماس با یک مشتری وقتی سودآور است که احتمال ریزشش بیشتر از $10/100 = 0.1$ باشد. آستانه‌ای که اعتبارسنجی متقابل پیدا کرده بسیار کمتر از ۰٫۵ است و سود روی دادهٔ آزمون به‌طور قابل توجهی بیشتر می‌شود. این روش درست تبدیل «امتیاز مدل» به «تصمیم کسب‌وکار» است (در پروژهٔ پایانی دوباره از آن استفاده می‌کنیم).

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

ویژگی‌های جدیدی بسازید (مثلاً نسبت `Balance` به `EstimatedSalary` یا اینکه موجودی صفر است یا نه) و ببینید Average Precision بهتر می‌شود یا نه.

</div>

In [5]:
def add_features(df):
    df = df.copy()
    df["BalanceSalaryRatio"] = df["Balance"] / df["EstimatedSalary"]
    df["ZeroBalance"] = (df["Balance"] == 0).astype(int)
    df["ProductsPerYear"] = df["NumOfProducts"] / (df["Tenure"] + 1)
    return df

new_numeric = numeric + ["BalanceSalaryRatio", "ZeroBalance", "ProductsPerYear"]
for name, cols, data in [("original features", numeric, X_train), ("with new features", new_numeric, add_features(X_train))]:
    prep = ColumnTransformer([("num", StandardScaler(), cols), ("cat", OneHotEncoder(drop="if_binary"), categorical)])
    for model_name, model in [("LogReg", LogisticRegression(max_iter=1000)),
                              ("HistGB", HistGradientBoostingClassifier(random_state=42))]:
        res = cross_validate(Pipeline([("prep", prep), ("model", model)]), data, y_train, cv=cv,
                             scoring="average_precision", n_jobs=-1)
        print(f"{model_name:<7} {name:<18} CV average precision = {res['test_score'].mean():.3f}")

LogReg  original features  CV average precision = 0.469


HistGB  original features  CV average precision = 0.692
LogReg  with new features  CV average precision = 0.470


HistGB  with new features  CV average precision = 0.689


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

هیچ‌کدام از این ویژگی‌ها کمک محسوسی نمی‌کنند. برای HistGradientBoosting انتظارش را داشتیم: درخت‌ها خودشان با چند تقسیم الگوهایی مثل «موجودی صفر» را پیدا می‌کنند. برای رگرسیون لجستیک هم بهبود ناچیز است، چون مشکل اصلی آن رابطهٔ **غیرخطی** سن و تعداد محصول با ریزش است، نه این نسبت‌ها. درس مهم: هر ایدهٔ مهندسی ویژگی را باید با اعتبارسنجی متقابل آزمود؛ ایده‌ای که منطقی به نظر می‌رسد، لزوماً اطلاعات جدیدی به مدل نمی‌دهد. (برای نمونه، افزودن `Age` به توان دو یا تبدیل `NumOfProducts` به ویژگی دسته‌ای را امتحان کنید.)

</div>